# Class item count

Counts YOLOseg instances per class for **any** label folder you pick.

- Each non-empty line in a `.txt` label file = one instance. First token = class id (0-based).
- Class names come from a class-names file (`6class.txt` / `classes.txt`): line 1 = class id 0, ...

**How to use:** run the cells top to bottom. Cell&nbsp;1 opens a folder picker — choose the label folder, then keep running.

## 1. Pick the label folder

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import tkinter as tk
from tkinter import filedialog

_root = tk.Tk(); _root.withdraw()
_chosen = filedialog.askdirectory(title='Select label folder (.txt files)')
_root.destroy()

LABEL_DIR = Path(_chosen)
assert LABEL_DIR.is_dir(), 'No folder selected'
print('folder:', LABEL_DIR)

## 2. Load class names

Auto-finds `6class.txt` / `classes.txt` by searching upward from the chosen folder. Falls back to numeric ids if none is found.

In [ ]:
def find_class_file(label_dir):
    for parent in (label_dir, *label_dir.parents):
        for name in ('6class.txt', 'classes.txt'):
            if (parent / name).is_file():
                return parent / name
    return None

CLASS_FILE = find_class_file(LABEL_DIR)
if CLASS_FILE:
    names = [l.strip() for l in CLASS_FILE.read_text().splitlines() if l.strip()]
    id2name = {i: n for i, n in enumerate(names)}
    print('class file:', CLASS_FILE)
else:
    id2name = {}
    print('No class-names file found — using numeric ids.')
id2name

## 3. Count instances

In [ ]:
counts = Counter()
files = empty = 0
for txt in LABEL_DIR.glob('*.txt'):
    files += 1
    lines = [ln for ln in txt.read_text().splitlines() if ln.strip()]
    if not lines:
        empty += 1
    for ln in lines:
        counts[int(ln.split()[0])] += 1

print(f'label files: {files}  (empty: {empty})')
print(f'total instances: {sum(counts.values())}')

## 4. Table

In [ ]:
ids = sorted(set(id2name) | set(counts))
df = pd.DataFrame([
    {'class_id': cid,
     'class_name': id2name.get(cid, f'UNKNOWN_{cid}'),
     'count': counts.get(cid, 0)}
    for cid in ids
])
total = df['count'].sum()
df['pct'] = (df['count'] / total * 100).round(2) if total else 0.0
df

## 5. Bar chart

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df['class_name'], df['count'], color='steelblue')
ax.set_ylabel('instances')
ax.set_title(f'Class item count — {LABEL_DIR.name}')
for i, v in enumerate(df['count']):
    ax.text(i, v, str(v), ha='center', va='bottom')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 6. Save results

Saves `class_counts_<folder>.csv` and `.png` next to the selected label folder.

In [ ]:
stem = f'class_counts_{LABEL_DIR.name}'
out_csv = LABEL_DIR.parent / f'{stem}.csv'
out_png = LABEL_DIR.parent / f'{stem}.png'
df.to_csv(out_csv, index=False)
fig.savefig(out_png, dpi=120)
print('saved:', out_csv)
print('saved:', out_png)